In [ ]:
import sys  # lets us change where Python looks for modules
sys.path.insert(0, "../src")  # so the notebook can import our files in src/
sys.path.insert(0, "..")  # project root too, because train.py now imports from src.

import pandas as pd  # reading the results CSV
import matplotlib.pyplot as plt  # making the per-category plot
from train import ROOT, log_run, summarize  # our helpers from step 3

by_fold = pd.read_csv(ROOT / "results" / "all_models_by_fold.csv")  # every model's score per fold and category

for model_name, df in by_fold.groupby("model"):  # loop over each model's rows
    metrics = summarize(df)  # average mae, rmse, smape, mase over all folds and categories
    per_cat = df.groupby("category")[["mae", "mase"]].mean()  # average MAE and MASE per sub-category
    for cat, row in per_cat.iterrows():  # loop over each sub-category
        metrics[f"mae_{cat}"] = float(row["mae"])  # e.g. "mae_Phones" as its own metric
        metrics[f"mase_{cat}"] = float(row["mase"])  # e.g. "mase_Phones" as its own metric

    fig, ax = plt.subplots(figsize=(8, 4))  # new empty figure for this model
    per_cat["mae"].sort_values().plot.barh(ax=ax)  # horizontal bars, best category at the bottom
    ax.set_title(f"{model_name}: MAE per sub-category")  # plot title with the model name
    ax.set_xlabel("MAE")  # x-axis label
    fig.tight_layout()  # stop labels from being cut off

    params = {  # settings that describe this run
        "model": model_name,  # which model
        "n_folds": int(df["fold"].nunique()),  # number of walk-forward folds
        "horizon_weeks": 8,  # forecast horizon used in every fold
        "n_categories": int(df["category"].nunique()),  # 9 for most models, 4 for SARIMA
        "stage": "relog_day2_to_4",  # marks these as re-logged older results
    }

    log_run(model_name, params, metrics, figure=fig, results_df=df)  # send everything to MLflow
    print(f"{model_name:25s} cats={params['n_categories']}  MAE={metrics['mae']:.0f}  MASE={metrics['mase']:.2f}")  # quick summary line

lgbm_global               cats=9  MAE=867  MASE=1.07
lgbm_per_category         cats=9  MAE=986  MASE=1.23
moving_avg_4              cats=9  MAE=987  MASE=1.23
naive                     cats=9  MAE=1133  MASE=1.45
sarima                    cats=4  MAE=1171  MASE=0.98
seasonal_naive            cats=9  MAE=1127  MASE=1.38


In [2]:
sys.path.insert(0, "..")  # also add the project root, so "src." imports work like in notebooks 02-04
import time  # to time one evaluation run
import optuna  # the tuning library
from src.data import load_raw, make_weekly_table  # Day 1 loading functions
from src.models import DEFAULT_PARAMS, train_lgbm, recursive_forecast  # Day 3 LightGBM functions
from src.evaluate import evaluate  # Day 3 walk-forward scoring loop

raw = load_raw(ROOT / "data" / "superstore.csv")  # load and clean the raw CSV
weekly = make_weekly_table(raw)  # weekly table: 209 weeks x 9 sub-categories
tune_data = weekly.iloc[:-40]  # drop the last 40 weeks (the 5 test folds) so tuning never sees them


def make_forecast_fn(params):  # builds a forecast function that uses the given LightGBM settings
    def forecast_fn(train, horizon):  # same shape as global_forecast: (train table, horizon) in, forecast out
        model = train_lgbm(train, params)  # train the global model with these settings
        return recursive_forecast(model, train, horizon)  # forecast all 9 categories week by week
    return forecast_fn  # hand the function back to evaluate()


def objective(trial):  # Optuna calls this once per trial and tries to make the return value smaller
    params = {  # LightGBM settings that Optuna picks for this trial
        "n_estimators": trial.suggest_int("n_estimators", 100, 800, step=50),  # number of trees
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),  # step size, searched on a log scale
        "num_leaves": trial.suggest_int("num_leaves", 4, 31),  # tree size (small, our data is small)
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),  # minimum rows per leaf, limits overfitting
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),  # share of rows each tree sees
        "subsample_freq": 1,  # turn row sampling on (re-sample every tree)
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),  # share of features each tree sees
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),  # L2 penalty, shrinks extreme leaf values
        "random_state": 42,  # fixed seed so a trial is repeatable
        "verbose": -1,  # hide LightGBM messages
    }
    res = evaluate(tune_data, make_forecast_fn(params), "lgbm_tuning", n_folds=3)  # 3 folds x 8 weeks, before the test weeks
    return float(res["mase"].mean())  # average MASE over folds and categories: the score to minimize


start = time.time()  # start the timer
ref = evaluate(tune_data, make_forecast_fn(DEFAULT_PARAMS), "lgbm_default", n_folds=3)  # Day 3 default settings on the tuning folds
print(f"default params tuning MASE: {ref['mase'].mean():.3f}")  # the score Optuna has to beat
print(f"one evaluation took {time.time() - start:.1f} s")  # tells us how long each trial will take

default params tuning MASE: 1.028
one evaluation took 2.8 s


In [3]:
import mlflow  # experiment tracking
from train import setup_mlflow  # our helper that points MLflow at mlflow.db

optuna.logging.set_verbosity(optuna.logging.WARNING)  # hide Optuna's long per-trial messages, we print our own
setup_mlflow()  # make sure runs go to the sales-forecasting experiment


def log_trial(study, trial):  # Optuna calls this after every trial finishes
    with mlflow.start_run(run_name=f"trial_{trial.number:02d}", nested=True):  # one child run per trial, inside the parent run
        mlflow.log_params(trial.params)  # the settings this trial tried
        mlflow.log_metric("tuning_mase", trial.value)  # the score it got on the tuning folds
    best = " <- new best" if trial.number == study.best_trial.number else ""  # flag trials that set a new best score
    print(f"trial {trial.number:02d}  MASE {trial.value:.3f}{best}")  # one short line per trial


with mlflow.start_run(run_name="optuna_lgbm_global"):  # parent run that groups all 40 trials
    sampler = optuna.samplers.TPESampler(seed=42)  # smart search that learns from earlier trials; seed makes it repeatable
    study = optuna.create_study(direction="minimize", sampler=sampler)  # we want the lowest MASE
    study.optimize(objective, n_trials=40, callbacks=[log_trial])  # run 40 trials, logging each one
    mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})  # save the winning settings on the parent run
    mlflow.log_metric("best_tuning_mase", study.best_value)  # save the best score on the parent run
    mlflow.log_metric("default_tuning_mase", float(ref["mase"].mean()))  # save the default score next to it for comparison

print(f"\nbest tuning MASE: {study.best_value:.3f}  (default was {ref['mase'].mean():.3f})")  # tuned vs default
print(study.best_params)  # the winning settings

trial 00  MASE 1.117 <- new best
trial 01  MASE 1.045 <- new best
trial 02  MASE 0.980 <- new best
trial 03  MASE 0.975 <- new best
trial 04  MASE 0.977
trial 05  MASE 0.975
trial 06  MASE 1.049
trial 07  MASE 1.067
trial 08  MASE 0.982
trial 09  MASE 0.962 <- new best
trial 10  MASE 0.986
trial 11  MASE 0.966
trial 12  MASE 0.978
trial 13  MASE 0.960 <- new best
trial 14  MASE 1.015
trial 15  MASE 0.978
trial 16  MASE 0.973
trial 17  MASE 0.971
trial 18  MASE 0.987
trial 19  MASE 1.036
trial 20  MASE 0.971
trial 21  MASE 0.982
trial 22  MASE 0.986
trial 23  MASE 0.967
trial 24  MASE 0.986
trial 25  MASE 0.962
trial 26  MASE 0.975
trial 27  MASE 0.962
trial 28  MASE 0.966
trial 29  MASE 0.966
trial 30  MASE 0.965
trial 31  MASE 0.976
trial 32  MASE 0.976
trial 33  MASE 0.979
trial 34  MASE 0.978
trial 35  MASE 0.969
trial 36  MASE 0.988
trial 37  MASE 0.979
trial 38  MASE 1.003
trial 39  MASE 0.971

best tuning MASE: 0.960  (default was 1.028)
{'n_estimators': 450, 'learning_rate': 0.0

In [4]:
TUNED_PARAMS = {**study.best_params, "subsample_freq": 1, "random_state": 42, "verbose": -1}  # Optuna's winners plus the fixed settings
res_tuned = evaluate(weekly, make_forecast_fn(TUNED_PARAMS), "lgbm_global_tuned")  # all 209 weeks, the same 5 folds as Days 2-4

metrics = summarize(res_tuned)  # average mae, rmse, smape, mase over folds and categories
per_cat = res_tuned.groupby("category")[["mae", "mase"]].mean()  # average per sub-category
for cat, row in per_cat.iterrows():  # loop over the 9 sub-categories
    metrics[f"mae_{cat}"] = float(row["mae"])  # e.g. "mae_Phones"
    metrics[f"mase_{cat}"] = float(row["mase"])  # e.g. "mase_Phones"

fig, ax = plt.subplots(figsize=(8, 4))  # new figure
per_cat["mae"].sort_values().plot.barh(ax=ax)  # MAE per sub-category, best at the bottom
ax.set_title("lgbm_global_tuned: MAE per sub-category")  # title
ax.set_xlabel("MAE")  # x-axis label
fig.tight_layout()  # keep labels inside the figure

params = {**TUNED_PARAMS, "model": "lgbm_global_tuned", "n_folds": 5, "horizon_weeks": 8, "n_categories": 9, "stage": "day5_tuned"}  # settings to log
log_run("lgbm_global_tuned", params, metrics, figure=fig, results_df=res_tuned)  # send the run to MLflow

res_tuned.to_csv(ROOT / "results" / "tuned_scores_by_fold.csv", index=False)  # save it; older result files stay untouched

compare = pd.concat([by_fold[by_fold["model"] != "sarima"], res_tuned], ignore_index=True)  # all 9-category models plus the tuned one (SARIMA left out, only 4 categories)
table = compare.groupby("model")[["mae", "rmse", "smape", "mase"]].mean().sort_values("mae").round(2)  # average per model, best MAE first
print(table)  # main comparison table

mae_cat = compare.pivot_table(index="category", columns="model", values="mae").round(0)  # MAE per category, one column per model
print(mae_cat[["moving_avg_4", "lgbm_global", "lgbm_global_tuned"]])  # baseline vs default vs tuned, side by side

                       mae     rmse  smape  mase
model                                           
lgbm_global_tuned   863.01  1133.35  76.30  1.06
lgbm_global         867.22  1132.52  75.92  1.07
lgbm_per_category   986.14  1260.29  89.54  1.23
moving_avg_4        987.32  1205.53  82.88  1.23
seasonal_naive     1127.14  1460.40  98.48  1.38
naive              1133.06  1384.60  91.14  1.45
model        moving_avg_4  lgbm_global  lgbm_global_tuned
category                                                 
Accessories         874.0        850.0              859.0
Appliances          726.0        530.0              590.0
Binders            1320.0       1182.0             1164.0
Chairs             1602.0       1204.0             1209.0
Furnishings         541.0        396.0              387.0
Paper               345.0        312.0              280.0
Phones             1275.0       1286.0             1328.0
Storage            1166.0        979.0              919.0
Tables             1037.0   